# 20_silver_dims — Bronze dimensions to Silver

Loads current user and track state with hash-guarded SCD1 MERGEs.

In [ ]:
# MAGIC %run ./00_config


In [ ]:
def latest_snapshot(table, key):
    window=Window.partitionBy(key).orderBy(F.col('load_timestamp').desc(),F.col('_batch_id').desc())
    return spark.table(table).withColumn('_rn',F.row_number().over(window)).where('_rn = 1').drop('_rn')

def merge_dimensions(source, target, key, columns):
    source.createOrReplaceTempView('silver_dimension_source')
    update=', '.join([f't.`{name}` = s.`{name}`' for name in columns if name not in {key,'_row_hash'}] + ['t._row_hash = s._row_hash', 't._batch_id = s._batch_id', 't.load_timestamp = s.load_timestamp'])
    insert_columns=', '.join(f'`{name}`' for name in columns)
    insert_values=', '.join(f's.`{name}`' for name in columns)
    spark.sql(f'''MERGE INTO {table_ref(target)} t USING silver_dimension_source s ON t.`{key}` = s.`{key}`
      WHEN MATCHED AND t._row_hash <> s._row_hash THEN UPDATE SET {update}
      WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})''')
    return execution_metrics(target)

started=datetime.now(timezone.utc)
try:
    users=(latest_snapshot(BRONZE_USERS,'user_id').select(F.trim('user_id').alias('user_id'),F.upper(F.trim('country')).alias('country'),F.trim('timezone').alias('timezone'),F.col('is_premium').cast('boolean').alias('is_premium'),F.lit(BATCH_ID).alias('_batch_id')).withColumn('_row_hash',F.sha2(F.to_json(F.struct('country','timezone','is_premium')),256)).withColumn('load_timestamp',F.current_timestamp()))
    user_columns=['user_id','country','timezone','is_premium','_row_hash','_batch_id','load_timestamp']
    user_metrics=merge_dimensions(users,SILVER_USERS,'user_id',user_columns)
    log_run('BRONZE_TO_SILVER',SILVER_USERS,f'{START_DATE}..{END_DATE}',started,'SUCCESS',users.count(),user_metrics['inserted'],user_metrics['updated'],user_metrics['deleted'])
except Exception as exc:
    log_run('BRONZE_TO_SILVER',SILVER_USERS,f'{START_DATE}..{END_DATE}',started,'FAILURE',error_message=f'{type(exc).__name__}: {exc}')
    raise

started=datetime.now(timezone.utc)
try:
    tracks=(latest_snapshot(BRONZE_CATALOG,'track_id').select(*[F.trim(F.col(name)).alias(name) if name in {'track_id','track_name','artist','genre'} else F.col(name).cast('int').alias(name) if name in {'release_year','duration_sec','popularity'} else F.col(name).cast('double').alias(name) if name in {'danceability','energy','valence','tempo_bpm'} else F.col(name) for name in ['track_id','track_name','artist','genre','release_year','duration_sec','popularity','danceability','energy','valence','tempo_bpm']],F.lit(BATCH_ID).alias('_batch_id'))
    business=['track_name','artist','genre','release_year','duration_sec','popularity','danceability','energy','valence','tempo_bpm']
    tracks=tracks.withColumn('_row_hash',F.sha2(F.to_json(F.struct(*business)),256)).withColumn('load_timestamp',F.current_timestamp())
    track_columns=['track_id']+business+['_row_hash','_batch_id','load_timestamp']
    track_metrics=merge_dimensions(tracks,SILVER_TRACKS,'track_id',track_columns)
    log_run('BRONZE_TO_SILVER',SILVER_TRACKS,f'{START_DATE}..{END_DATE}',started,'SUCCESS',tracks.count(),track_metrics['inserted'],track_metrics['updated'],track_metrics['deleted'])
except Exception as exc:
    log_run('BRONZE_TO_SILVER',SILVER_TRACKS,f'{START_DATE}..{END_DATE}',started,'FAILURE',error_message=f'{type(exc).__name__}: {exc}')
    raise
